# `contradiction_gate4.py` — Playground

Manual verification notebook for **Gate 4: Contradiction Detection** (Claude call 3 of 4).

| Function | Status | Notes |
|---|---|---|
| `detect_gate4_contradiction(candidate, gate3_result, market_context)` | ✅ built | One focused Claude question: does the market backdrop contradict an already-bullish entry? |

**Output:** `{passed, contradiction_detected, contradiction_type, risk_level, reason, action, market_context}`

Gate 4 runs **after** Gates 1–3 already cleared the candidate. Its only job is the **gray zone** Gate 1's hard thresholds wave through — so it screens just the two contradictions Gate 1 cannot catch:

- `DIVERGENCE` — the stock pushes higher while its sector / the broad market weakens (Gate 1 never compares stock vs market).
- `BROAD_RISK_OFF` — VIX, SPY **and** the sector each only *mildly* negative (all below Gate 1's hard limits) but together leaning risk-off.

**The gate never fetches.** The caller passes `market_context` in (the pipeline assembles it from Gate 1's already-fetched data; here we call `get_market_context` ourselves). Claude returns a `contradiction_type` + `risk_level`; the **action is derived in code** from the risk level:

| risk_level | action | passed |
|---|---|---|
| HIGH | `BLOCK` | False |
| MEDIUM / LOW | `FLAG_FOR_REVIEW` | False |
| NONE (no contradiction) | `PASS` | True |
| LLM unavailable | `BLOCK` (`reason='llm_unavailable'`) | False |

**Standard used:** `helpers/llm/client.build_agent` + `run_agent` (cheap Haiku default). Live calls need `ANTHROPIC_API_KEY` in `.env`.

In [2]:
import sys
import pathlib

# Anchor paths to this notebook's folder — kernel cwd may be repo root OR gate4_contradiction/.
gate4_dir = pathlib.Path('.').resolve()
if not (gate4_dir / 'contradiction_gate4.py').exists():
    gate4_dir = pathlib.Path('backend/02_intelligence/gate4_contradiction').resolve()

intelligence_dir = gate4_dir.parent
gate3_dir        = intelligence_dir / 'gate3_sentiment'

for p in [str(intelligence_dir), str(gate3_dir), str(gate4_dir)]:
    if p not in sys.path:
        sys.path.insert(0, p)

from contradiction_gate4 import detect_gate4_contradiction
from sentiment_gate3 import evaluate_gate3_sentiment
from helpers.fetchers.market import get_market_context
from helpers.fetchers.news import fetch_news

candidate = {'ticker': 'NVDA', 'sector': 'Electronic Technology'}
bullish   = {'direction': 'BULLISH', 'confidence': 8}  # synthetic fixture sections only

---
## Happy path — calm market → PASS

A synthetic calm backdrop: low VIX, SPY and sector both green. Nothing contradicts the bullish entry — expect `action='PASS'`, `contradiction_detected=False`.

In [3]:
calm = {
    'vix':    {'level': 14.5, 'change_pct_today': -0.01,  'prior_close': 14.6},
    'spy':    {'price': 540.0, 'change_pct_today': 0.004, 'prior_close': 537.8},
    'sector': {'etf_ticker': 'XLK', 'price': 230.0, 'change_pct_today': 0.006, 'prior_close': 228.6},
    'hours_to_next_macro': 30.0,
}

detect_gate4_contradiction(candidate, bullish, calm)

[gate4] NVDA: passed — no contradiction (risk=NONE)


{'passed': True,
 'contradiction_detected': False,
 'contradiction_type': 'none',
 'risk_level': 'NONE',
 'reason': 'NONE',
 'action': 'PASS',
 'market_context': {'vix': {'level': 14.5,
   'change_pct_today': -0.01,
   'prior_close': 14.6},
  'spy': {'price': 540.0, 'change_pct_today': 0.004, 'prior_close': 537.8},
  'sector': {'etf_ticker': 'XLK',
   'price': 230.0,
   'change_pct_today': 0.006,
   'prior_close': 228.6},
  'hours_to_next_macro': 30.0}}

---
## Variation — gray-zone risk-off → FLAG or BLOCK

VIX 26 (below Gate 1's 30), SPY −1.2% (above −1.5%), sector −1.8% (above −2%) — **every signal passes Gate 1**, but together they lean risk-off. Expect `broad_risk_off` with a non-PASS action. This is the accumulation case Gate 4 exists for.

In [4]:
risk_off = {
    'vix':    {'level': 26.0, 'change_pct_today': 0.18,  'prior_close': 22.0},
    'spy':    {'price': 528.0, 'change_pct_today': -0.012, 'prior_close': 534.4},
    'sector': {'etf_ticker': 'XLK', 'price': 221.0, 'change_pct_today': -0.018, 'prior_close': 225.0},
    'hours_to_next_macro': 30.0,
}

detect_gate4_contradiction(candidate, bullish, risk_off)

[gate4] NVDA: FLAG_FOR_REVIEW — broad_risk_off risk=MEDIUM: VIX spiked 18% today, SPY is down 1.20%, and the tech sector (XLK) is down 1.80%—together these paint a mild but clear risk-off backdrop that undermines a fresh bullish entry despite NVDA's own momentum strength.


{'passed': False,
 'contradiction_detected': True,
 'contradiction_type': 'broad_risk_off',
 'risk_level': 'MEDIUM',
 'reason': "VIX spiked 18% today, SPY is down 1.20%, and the tech sector (XLK) is down 1.80%—together these paint a mild but clear risk-off backdrop that undermines a fresh bullish entry despite NVDA's own momentum strength.",
 'action': 'FLAG_FOR_REVIEW',
 'market_context': {'vix': {'level': 26.0,
   'change_pct_today': 0.18,
   'prior_close': 22.0},
  'spy': {'price': 528.0, 'change_pct_today': -0.012, 'prior_close': 534.4},
  'sector': {'etf_ticker': 'XLK',
   'price': 221.0,
   'change_pct_today': -0.018,
   'prior_close': 225.0},
  'hours_to_next_macro': 30.0}}

---
## Variation — divergence → FLAG or BLOCK

The sector is clearly weak while we hold a strong bullish thesis on the stock — the stock is diverging from its sector. Expect `divergence` (the contradiction no other gate can see).

In [5]:
divergent = {
    'vix':    {'level': 17.0, 'change_pct_today': 0.02,  'prior_close': 16.7},
    'spy':    {'price': 535.0, 'change_pct_today': -0.002, 'prior_close': 536.1},
    'sector': {'etf_ticker': 'XLK', 'price': 222.0, 'change_pct_today': -0.016, 'prior_close': 225.6},
    'hours_to_next_macro': 40.0,
}

detect_gate4_contradiction(candidate, bullish, divergent)

[gate4] NVDA: FLAG_FOR_REVIEW — divergence risk=MEDIUM: NVDA is a bullish momentum candidate while its sector (XLK) is down 1.60% and SPY is slightly negative, creating a divergence that risks mean reversion if sector weakness persists.


{'passed': False,
 'contradiction_detected': True,
 'contradiction_type': 'divergence',
 'risk_level': 'MEDIUM',
 'reason': 'NVDA is a bullish momentum candidate while its sector (XLK) is down 1.60% and SPY is slightly negative, creating a divergence that risks mean reversion if sector weakness persists.',
 'action': 'FLAG_FOR_REVIEW',
 'market_context': {'vix': {'level': 17.0,
   'change_pct_today': 0.02,
   'prior_close': 16.7},
  'spy': {'price': 535.0, 'change_pct_today': -0.002, 'prior_close': 536.1},
  'sector': {'etf_ticker': 'XLK',
   'price': 222.0,
   'change_pct_today': -0.016,
   'prior_close': 225.6},
  'hours_to_next_macro': 40.0}}

---
## Real market — multiple tickers

The genuine end-to-end path (live `fetch_news` → Gate 3 sentiment → live `get_market_context(sector)` → Gate 4), not synthetic fixtures. Three tickers across different sectors — NVIDIA (tech), JPMorgan (finance), Exxon (energy) — so you can see real variation in sentiment and market backdrop. Gate 4 only runs when Gate 3 passes, matching the pipeline. Results change with the news window and the live tape.

In [6]:
# Real pipeline shape per ticker: fetch news once, Gate 3 assesses sentiment, then Gate 4
# judges the live market backdrop using Gate 3's direction + confidence (not a fixture).
for tkr, name, sector in [('NVDA', 'NVIDIA Corporation',      'Electronic Technology'),
                          ('JPM',  'JPMorgan Chase & Co.',    'Finance'),
                          ('XOM',  'Exxon Mobil Corporation', 'Energy Minerals')]:
    candidate = {'ticker': tkr, 'company_name': name, 'sector': sector}

    headlines = fetch_news(tkr) or []
    g3 = evaluate_gate3_sentiment(candidate, headlines)
    if not g3['passed']:
        print(f"{tkr:<5} gate3 BLOCK       dir={g3['direction']:<8} conf={g3['confidence']} — skipping gate4")
        continue

    ctx = get_market_context(sector)
    if ctx is None:
        print(f'{tkr:<5} market context unavailable — skipping gate4')
        continue

    gate3_result = {'direction': g3['direction'], 'confidence': g3['confidence']}
    r = detect_gate4_contradiction(candidate, gate3_result, ctx)
    print(f"{tkr:<5} gate4 {r['action']:<12} dir={g3['direction']:<8} conf={g3['confidence']}  "
          f"type={r['contradiction_type']:<14} risk={r['risk_level']}")

[gate3] NVDA: BLOCKED — NEUTRAL conf=5: Mixed signals with outflows from Magnificent 7 stocks offsetting positive CEO leadership commentary and semiconductor sector strength, creating an ambiguous near-term momentum picture.
NVDA  gate3 BLOCK       dir=NEUTRAL  conf=5 — skipping gate4
[gate3] JPM: passed — BULLISH conf=7
[gate4] JPM: FLAG_FOR_REVIEW — divergence risk=MEDIUM: JPM is in a bullish sector (XLF +0.22%) while the broad market weakens (SPY -0.72%), creating relative strength that may snap back if market sentiment deteriorates further.
JPM   gate4 FLAG_FOR_REVIEW dir=BULLISH  conf=7  type=divergence     risk=MEDIUM
[gate3] XOM: BLOCKED — BEARISH conf=6: The first headline indicates key talent departure from Exxon's gas trading division, signaling potential operational challenges, while the second headline is irrelevant to XOM fundamentals.
XOM   gate3 BLOCK       dir=BEARISH  conf=6 — skipping gate4


---
## Free-play

Try your own market contexts, sectors, or sentiment confidences below.